### **Loading Partitions & Defining Feature Groups**

In [13]:
import pandas as pd
import numpy as np

# 1. Load the pre-split, leakage-free partitions
train_df = pd.read_csv("../data/processed/train_encounters.csv")
test_df = pd.read_csv("../data/processed/test_encounters.csv")

print(f"Loaded Train encounters: {train_df.shape}")
print(f"Loaded Test encounters:  {test_df.shape}")

# 2. Separate features (X) and binary target (y)
target_col = "target_readmit_30d"

# Non-predictive columns to exclude:
# - Identifiers (encounter_id, patient_nbr)
# - Original multiclass label (readmitted)
# - Target itself (target_readmit_30d)
cols_to_drop = ["encounter_id", "patient_nbr", "readmitted", target_col]

X_train = train_df.drop(columns=cols_to_drop)
y_train = train_df[target_col]

X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df[target_col]

print(f"\nFeature matrix X_train shape: {X_train.shape}")
print(f"Feature matrix X_test shape:  {X_test.shape}")

Loaded Train encounters: (79478, 50)
Loaded Test encounters:  (19862, 50)

Feature matrix X_train shape: (79478, 46)
Feature matrix X_test shape:  (19862, 46)


### **Isolating Numerical and Categorical Pipelines**

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Explicitly identify numerical features
num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# 2. Identify categorical features (remaining object or integer code columns)
cat_cols = [c for c in X_train.columns if c not in num_cols]

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols[:5]} ... (+{len(cat_cols)-5} more)")

# 3. Build sub-pipelines
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 4. Combine into a unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols)
    ],
    remainder="drop"
)

# Test fit-transform strictly on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nSuccessfully fitted preprocessor!")
print(f"Transformed X_train shape: {X_train_processed.shape}")
print(f"Transformed X_test shape:  {X_test_processed.shape}")

Numerical features (8): ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Categorical features (38): ['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id'] ... (+33 more)

Successfully fitted preprocessor!
Transformed X_train shape: (79446, 273)
Transformed X_test shape:  (19676, 273)


### **Establishing the Logistic Regression Baseline**

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# 1. Initialize Baseline Logistic Regression with balanced weighting
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
    solver="lbfgs"
)

# 2. Train baseline model on preprocessed training data
log_reg.fit(X_train_processed, y_train)

# 3. Generate predictions and predicted probabilities on held-out test set
y_pred_lr = log_reg.predict(X_test_processed)
y_prob_lr = log_reg.predict_proba(X_test_processed)[:, 1]

# 4. Calculate key metrics
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr)

print("================ Logistic Regression Baseline Results ================")
print(f"ROC-AUC Score: {lr_roc_auc:.4f}")
print("\nConfusion Matrix [ [TN, FP], [FN, TP] ]:")
print(lr_cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=["No Early Readmit (0)", "Readmitted <30d (1)"]))

================ Logistic Regression Baseline Results ================
ROC-AUC Score: 0.6711

Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[12029  5430]
 [ 1000  1217]]

Classification Report:
                      precision    recall  f1-score   support

No Early Readmit (0)       0.92      0.69      0.79     17459
 Readmitted <30d (1)       0.18      0.55      0.27      2217

            accuracy                           0.67     19676
           macro avg       0.55      0.62      0.53     19676
        weighted avg       0.84      0.67      0.73     19676



Training Candidate Models (Decision Tree & Random Forest)

In [5]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Decision Tree Comparison Baseline
# We constrain max_depth to prevent severe overfitting on the 273 features
dt_model = DecisionTreeClassifier(
    max_depth=6,
    class_weight="balanced",
    random_state=42
)
dt_model.fit(X_train_processed, y_train)

y_pred_dt = dt_model.predict(X_test_processed)
y_prob_dt = dt_model.predict_proba(X_test_processed)[:, 1]

dt_roc_auc = roc_auc_score(y_test, y_prob_dt)
print(f"Decision Tree (max_depth=6) ROC-AUC: {dt_roc_auc:.4f}")

# 2. Primary Candidate Model: Random Forest Classifier
# n_estimators=150 provides sufficient ensemble stability
# max_depth=12 and min_samples_leaf=20 prevent individual trees from memorizing rare encounter noise
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

print("\nTraining Random Forest Classifier (this may take 20-40 seconds)...")
rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)
y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

rf_roc_auc = roc_auc_score(y_test, y_prob_rf)
print(f"Random Forest Classifier ROC-AUC:    {rf_roc_auc:.4f}")

Decision Tree (max_depth=6) ROC-AUC: 0.6523

Training Random Forest Classifier (this may take 20-40 seconds)...
Random Forest Classifier ROC-AUC:    0.6734


In [6]:
from sklearn.metrics import recall_score, precision_score, f1_score

# Construct unified comparison table
models_summary = pd.DataFrame({
    "Model": ["Logistic Regression (Baseline)", "Decision Tree (max_depth=6)", "Random Forest (Primary)"],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_dt),
        roc_auc_score(y_test, y_prob_rf)
    ],
    "Recall (Class 1)": [
        recall_score(y_test, y_pred_lr, pos_label=1),
        recall_score(y_test, y_pred_dt, pos_label=1),
        recall_score(y_test, y_pred_rf, pos_label=1)
    ],
    "Precision (Class 1)": [
        precision_score(y_test, y_pred_lr, pos_label=1),
        precision_score(y_test, y_pred_dt, pos_label=1),
        precision_score(y_test, y_pred_rf, pos_label=1)
    ],
    "F1-Score (Class 1)": [
        f1_score(y_test, y_pred_lr, pos_label=1),
        f1_score(y_test, y_pred_dt, pos_label=1),
        f1_score(y_test, y_pred_rf, pos_label=1)
    ]
})

print("================ Comprehensive Model Benchmark Table ================")
print(models_summary.to_string(index=False))

print("\nRandom Forest Confusion Matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["No Early Readmit (0)", "Readmitted <30d (1)"]))

================ Comprehensive Model Benchmark Table ================
                         Model  ROC-AUC  Recall (Class 1)  Precision (Class 1)  F1-Score (Class 1)
Logistic Regression (Baseline) 0.671119          0.548940             0.183090            0.274594
   Decision Tree (max_depth=6) 0.652350          0.535859             0.178835            0.268172
       Random Forest (Primary) 0.673426          0.556157             0.184470            0.277048

Random Forest Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[12008  5451]
 [  984  1233]]

Random Forest Classification Report:
                      precision    recall  f1-score   support

No Early Readmit (0)       0.92      0.69      0.79     17459
 Readmitted <30d (1)       0.18      0.56      0.28      2217

            accuracy                           0.67     19676
           macro avg       0.55      0.62      0.53     19676
        weighted avg       0.84      0.67      0.73     19676



In [7]:
# 1. Retrieve all one-hot feature names generated by ColumnTransformer
feature_names = preprocessor.get_feature_names_out()

# 2. Extract feature importances from the fitted Random Forest
importances = rf_model.feature_importances_

feat_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Top 15 Most Influential Features in Random Forest:")
print(feat_imp_df.head(15).to_string(index=False))

Top 15 Most Influential Features in Random Forest:
                         Feature  Importance
           num__number_inpatient    0.206545
 cat__discharge_disposition_id_1    0.069665
           num__number_emergency    0.050776
            num__num_medications    0.043900
cat__discharge_disposition_id_22    0.043360
         num__num_lab_procedures    0.040596
           num__time_in_hospital    0.039817
           num__number_diagnoses    0.039431
          num__number_outpatient    0.017335
             num__num_procedures    0.016436
 cat__discharge_disposition_id_3    0.016015
                 cat__insulin_No    0.010795
   cat__payer_code_Missing_Payer    0.009348
                cat__age_[50-60)    0.009133
 cat__discharge_disposition_id_6    0.008057


In [8]:
# Threshold optimization scan for Random Forest
thresholds = np.arange(0.35, 0.55, 0.02)
scan_results = []

for t in thresholds:
    y_pred_t = (y_prob_rf >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm_t.ravel()
    
    rec = recall_score(y_test, y_pred_t, pos_label=1)
    prec = precision_score(y_test, y_pred_t, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, pos_label=1)
    
    scan_results.append({
        "Threshold": round(t, 2),
        "Recall": round(rec, 4),
        "Precision": round(prec, 4),
        "F1-Score": round(f1, 4),
        "False Negatives": fn,
        "False Positives": fp,
        "True Positives": tp
    })

threshold_df = pd.DataFrame(scan_results)
print("================ Random Forest Threshold Optimization Scan ================")
print(threshold_df.to_string(index=False))

================ Random Forest Threshold Optimization Scan ================
 Threshold  Recall  Precision  F1-Score  False Negatives  False Positives  True Positives
      0.35  0.9802     0.1175    0.2099               44            16314            2173
      0.37  0.9626     0.1220    0.2165               83            15359            2134
      0.39  0.9314     0.1273    0.2240              152            14153            2065
      0.41  0.8877     0.1345    0.2336              249            12666            1968
      0.43  0.8290     0.1429    0.2438              379            11024            1838
      0.45  0.7654     0.1533    0.2554              520             9375            1697
      0.47  0.6964     0.1649    0.2666              673             7821            1544
      0.49  0.6058     0.1770    0.2740              874             6243            1343
      0.51  0.5038     0.1942    0.2804             1100             4634            1117
      0.53  0.4078     0

In [10]:
import joblib
import os

# 1. Construct unified inference pipeline combining preprocessor and trained estimator
final_readmission_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", rf_model)
])

# 2. Verify directory existence
os.makedirs("../app", exist_ok=True)
model_path = "../app/readmission_rf_pipeline.joblib"

# 3. Serialize artifact
joblib.dump(final_readmission_pipeline, model_path)

print(f"Unified model pipeline serialized successfully to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")

# 4. Smoke-test: load the pipeline back and test on raw X_test (before transformation)
loaded_pipeline = joblib.load(model_path)
sample_test_pred = loaded_pipeline.predict_proba(X_test.iloc[:5])[:, 1]
print("\nSmoke test validation: Pipeline runs on raw X_test DataFrame rows:")
print("Predicted probabilities for first 5 test encounters:", sample_test_pred.round(4))

Unified model pipeline serialized successfully to: ../app/readmission_rf_pipeline.joblib
File size: 7.09 MB

Smoke test validation: Pipeline runs on raw X_test DataFrame rows:
Predicted probabilities for first 5 test encounters: [0.4625 0.438  0.5387 0.3276 0.462 ]


In [11]:
# 1. Operationalize calibrated decision boundary (tau = 0.49)
OPERATIONAL_THRESHOLD = 0.49

y_pred_calibrated = (y_prob_rf >= OPERATIONAL_THRESHOLD).astype(int)

print(f"================ Random Forest Operational Evaluation (Threshold = {OPERATIONAL_THRESHOLD}) ================")
print(f"Final ROC-AUC: {rf_roc_auc:.4f}")
print("\nConfusion Matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, y_pred_calibrated))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_calibrated, target_names=["No Early Readmit (0)", "Readmitted <30d (1)"]))

================ Random Forest Operational Evaluation (Threshold = 0.49) ================
Final ROC-AUC: 0.6734

Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[11216  6243]
 [  874  1343]]

Classification Report:
                      precision    recall  f1-score   support

No Early Readmit (0)       0.93      0.64      0.76     17459
 Readmitted <30d (1)       0.18      0.61      0.27      2217

            accuracy                           0.64     19676
           macro avg       0.55      0.62      0.52     19676
        weighted avg       0.84      0.64      0.70     19676



In [12]:
# 1. Create audit dataframe combining demographic metadata with actuals and predictions
subgroup_df = test_df[["race", "gender", "age"]].copy()
subgroup_df["y_true"] = y_test.values
subgroup_df["y_prob"] = y_prob_rf
subgroup_df["y_pred"] = (y_prob_rf >= OPERATIONAL_THRESHOLD).astype(int)

# 2. Audit Recall and False Negative Rate across Race subgroups with N >= 100
print("================ Demographic Subgroup Performance Audit (Race) ================")
race_audit = []
for group, grp_data in subgroup_df.groupby("race"):
    total_positives = grp_data["y_true"].sum()
    if total_positives >= 30:  # Minimum clinical sample size threshold
        rec = recall_score(grp_data["y_true"], grp_data["y_pred"], pos_label=1, zero_division=0)
        prec = precision_score(grp_data["y_true"], grp_data["y_pred"], pos_label=1, zero_division=0)
        race_audit.append({
            "Subgroup": group,
            "Total Encounters": len(grp_data),
            "Actual Readmissions": total_positives,
            "Subgroup Recall": round(rec, 4),
            "Subgroup Precision": round(prec, 4)
        })

print(pd.DataFrame(race_audit).to_string(index=False))

================ Demographic Subgroup Performance Audit (Race) ================
       Subgroup  Total Encounters  Actual Readmissions  Subgroup Recall  Subgroup Precision
AfricanAmerican              3787                  447           0.5794              0.1812
      Caucasian             14629                 1653           0.6158              0.1747
       Hispanic               395                   37           0.6486              0.1765
   Unknown_Race               438                   40           0.4250              0.1889
